# Лабораторная работа 5: Дерево решений
## Простая реализация базовых функций

### Что реализовано:
1. Функция расчета критерия Джини
2. Функция расчета прироста информации (Information Gain)
3. Функция разбиения датасета в узле
4. Функция нахождения наилучшего разбиения
5. Функция построения дерева решений с критериями останова
6. Функция классификации объектов
7. Функция предсказания для датасета
8. Функция подсчета точности классификации
9. Функция комплексной оценки качества модели
10. Функция визуализации дерева
11. Класс Node (узел дерева)
12. Класс Leaf (лист дерева)
13. Базовый класс DecisionTree


In [ ]:
# Импорт функций
from tree_basics import gini, gain, split, find_best_split, build_tree, classify_object, predict, accuracy_metric, evaluate_model, print_tree, Node, Leaf, DecisionTree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Демонстрация критерия Джини
print("=== Критерий Джини ===")

# Тестовые примеры
test_labels = [
    [0, 0, 0, 0],           # чистый узел
    [0, 0, 1, 1],           # сбалансированный
    [0, 0, 0, 1],           # несбалансированный
    [0, 1, 2, 0, 1, 2],      # многоклассовый
    [0, 0, 1, 1, 1]
]

for i, labels in enumerate(test_labels):
    gini_val = gini(labels)
    print(f"Пример {i+1}: {labels} -> Gini = {gini_val:.3f}")

print("\n=== Классы ===")

# Демонстрация Leaf
data = [[1, 2], [2, 3]]
labels = [0, 1]
leaf = Leaf(data, labels)
print(f"Лист: предсказание = {leaf.prediction}")

# Демонстрация Node
left = Leaf([[0, 0]], [0])
right = Leaf([[1, 1]], [1])
node = Node(index=0, t=0.5, true_branch=right, false_branch=left)
print(f"Узел: признак {node.index}, порог {node.t}")

# Демонстрация DecisionTree
tree = DecisionTree€(max_depth=3)
print(f"Дерево создано с max_depth = {tree.max_depth}")


=== Критерий Джини ===
Пример 1: [0, 0, 0, 0] -> Gini = 0.000
Пример 2: [0, 0, 1, 1] -> Gini = 0.500
Пример 3: [0, 0, 0, 1] -> Gini = 0.375
Пример 4: [0, 1, 2, 0, 1, 2] -> Gini = 0.667
Пример 5: [0, 0, 1, 1, 1] -> Gini = 0.480

=== Классы ===
Лист: предсказание = 0
Узел: признак 0, порог 0.5
Дерево создано с max_depth = 3


In [12]:
print("\n=== Прирост информации (Information Gain) ===")

# Тестовые примеры для прироста информации
root_labels = [0, 0, 1, 1, 0, 1, 0, 1]  # корневой узел
root_gini_val = gini(root_labels)

test_splits = [
    ([0, 0, 0], [1, 1, 1, 0, 1]),  # хорошее разбиение
    ([0, 0, 1, 1], [0, 1, 0, 1]),  # среднее разбиение
    ([0, 0, 0, 0], [1, 1, 1, 1])   # плохое разбиение (одна ветвь чистая)
]

for i, (left, right) in enumerate(test_splits):
    ig_val = gain(left, right, root_gini_val)
    print(f"Разбиение {i+1}: лев={left}, прав={right}")
    print(f"  IG = {ig_val:.3f}")

print(f"\nКорневой Gini: {root_gini_val:.3f}")
print("• Чем выше IG, тем лучше разбиение")
print("• IG показывает насколько уменьшилась неопределенность после разбиения")



=== Прирост информации (Information Gain) ===
Разбиение 1: лев=[0, 0, 0], прав=[1, 1, 1, 0, 1]
  IG = 0.300
Разбиение 2: лев=[0, 0, 1, 1], прав=[0, 1, 0, 1]
  IG = 0.000
Разбиение 3: лев=[0, 0, 0, 0], прав=[1, 1, 1, 1]
  IG = 0.500

Корневой Gini: 0.500
• Чем выше IG, тем лучше разбиение
• IG показывает насколько уменьшилась неопределенность после разбиения


In [23]:
print("\n=== Разбиение датасета в узле ===")

# Создаем тестовый датасет
test_data = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 1.0],
    [4.0, 4.0],
    [0.5, 1.5]
])
test_labels = np.array([0, 0, 1, 1, 0])

print("Исходный датасет:")
print(f"Данные:\n{test_data}")
print(f"Метки: {test_labels}")

# Разбиение по колонке 0 (первый признак) с порогом 2.0
true_data, false_data, true_labels, false_labels = split(test_data, test_labels, column_index=0, t=2.0)

print(f"\nРазбиение по колонке 0, порог = 2.0:")
print(f"Левая ветвь (≤ 2.0):")
print(f"  Данные: {len(true_data)} объектов")
print(f"  Признаки:\n{true_data}")
print(f"  Метки: {true_labels}")

print(f"\nПравая ветвь (> 2.0):")
print(f"  Данные: {len(false_data)} объектов")
print(f"  Признаки:\n{false_data}")
print(f"  Метки: {false_labels}")

# Проверка корректности разбиения
print(f"\nПроверка корректности:")
print(f"Исходное количество объектов: {len(test_data)}")
print(f"Левая + правая: {len(true_data)} + {len(false_data)} = {len(true_data) + len(false_data)}")
print(f"Все метки учтены: {len(true_labels)} + {len(false_labels)} = {len(true_labels) + len(false_labels)}")



=== Разбиение датасета в узле ===
Исходный датасет:
Данные:
[[1.  2. ]
 [2.  3. ]
 [3.  1. ]
 [4.  4. ]
 [0.5 1.5]]
Метки: [0 0 1 1 0]

Разбиение по колонке 0, порог = 2.0:
Левая ветвь (≤ 2.0):
  Данные: 3 объектов
  Признаки:
[[1.  2. ]
 [2.  3. ]
 [0.5 1.5]]
  Метки: [0 0 0]

Правая ветвь (> 2.0):
  Данные: 2 объектов
  Признаки:
[[3. 1.]
 [4. 4.]]
  Метки: [1 1]

Проверка корректности:
Исходное количество объектов: 5
Левая + правая: 3 + 2 = 5
Все метки учтены: 3 + 2 = 5


In [24]:
print("\n=== Нахождение наилучшего разбиения ===")

# Создаем более сложный датасет для тестирования
np.random.seed(42)
complex_data = np.random.randn(20, 3)  # 20 объектов, 3 признака
complex_labels = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1])

print("Датасет для тестирования:")
print(f"Размер: {complex_data.shape}")
print(f"Метки: {complex_labels}")
print(f"Распределение классов: класс 0 - {np.sum(complex_labels == 0)}, класс 1 - {np.sum(complex_labels == 1)}")

# Находим наилучшее разбиение
best_gain, best_t, best_index = find_best_split(complex_data, complex_labels)

print("Результаты поиска наилучшего разбиения:")
print(f"Наилучший признак: {best_index}")
print(f"Оптимальный порог: {best_t}")
print(f"Максимальный прирост информации: {best_gain:.4f}")

# Проверяем разбиение вручную
if best_t is not None:
    true_data, false_data, true_labels, false_labels = split(complex_data, complex_labels, best_index, best_t)

    print("Проверка найденного разбиения:")
    print(f"Левая ветвь (≤ {best_t}): {len(true_labels)} объектов, метки: {true_labels}")
    print(f"Правая ветвь (> {best_t}): {len(false_labels)} объектов, метки: {false_labels}")

    # Рассчитываем прирост информации вручную для проверки
    root_gini = gini(complex_labels)
    manual_gain = gain(true_labels, false_labels, root_gini)

    print(f"Ручной расчет прироста: {manual_gain:.4f}")
    print(f"Совпадает с результатом: {abs(manual_gain - best_gain) < 1e-10}")

print("Анализ:")
print("• Функция перебрала все признаки и пороги")
print("• Учитывает ограничение min_samples_leaf = 3")
print("• Находит разбиение с максимальным приростом информации")



=== Нахождение наилучшего разбиения ===
Датасет для тестирования:
Размер: (20, 3)
Метки: [0 0 0 0 0 1 1 1 1 1 0 0 1 1 0 1 0 1 0 1]
Распределение классов: класс 0 - 10, класс 1 - 10
Результаты поиска наилучшего разбиения:
Наилучший признак: 0
Оптимальный порог: 0.2088635950047554
Максимальный прирост информации: 0.1875
Проверка найденного разбиения:
Левая ветвь (≤ 0.2088635950047554): 12 объектов, метки: [1 1 1 1 0 0 1 1 0 1 1 1]
Правая ветвь (> 0.2088635950047554): 8 объектов, метки: [0 0 0 0 0 1 0 0]
Ручной расчет прироста: 0.1875
Совпадает с результатом: True
Анализ:
• Функция перебрала все признаки и пороги
• Учитывает ограничение min_samples_leaf = 3
• Находит разбиение с максимальным приростом информации


In [25]:
print("\n=== Полное дерево решений ===")

# Создаем простой датасет для обучения дерева
simple_data = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 1.0],
    [4.0, 4.0],
    [0.5, 1.5],
    [1.5, 2.5],
    [2.5, 0.5],
    [3.5, 3.5]
])
simple_labels = np.array([0, 0, 1, 1, 0, 0, 1, 1])

print("Обучающий датасет:")
print(f"Размер: {simple_data.shape}")
print(f"Данные:\n{simple_data}")
print(f"Метки: {simple_labels}")

# Строим дерево решений
print("\nПостроение дерева решений...")
my_tree = build_tree(simple_data, simple_labels)

print("Дерево построено!")

# Визуализируем дерево
print("\nСтруктура дерева:")
print_tree(my_tree)

# Тестовые данные для предсказания
test_data = np.array([
    [1.5, 2.5],  # должен быть класс 0
    [3.2, 2.8],  # должен быть класс 1
    [0.8, 1.2],  # должен быть класс 0
])

print("\nТестовые данные:")
print(f"Данные:\n{test_data}")

# Делаем предсказания
predictions = predict(test_data, my_tree)
print(f"Предсказания: {predictions}")

# Проверяем предсказания вручную
print("\nПроверка предсказаний:")
for i, obj in enumerate(test_data):
    prediction = classify_object(obj, my_tree)
    print(f"Объект {obj} -> Класс {prediction}")

print("\nАнализ:")
print("• Дерево решений успешно построено с помощью рекурсии")
print("• Каждый узел содержит условие разбиения")
print("• Листья содержат финальные предсказания")
print("• Функции классификации и предсказания работают корректно")



=== Полное дерево решений ===
Обучающий датасет:
Размер: (8, 2)
Данные:
[[1.  2. ]
 [2.  3. ]
 [3.  1. ]
 [4.  4. ]
 [0.5 1.5]
 [1.5 2.5]
 [2.5 0.5]
 [3.5 3.5]]
Метки: [0 0 1 1 0 0 1 1]

Построение дерева решений...
Дерево построено!

Структура дерева:
Индекс 0 <= 2.0
--> True:
  Прогноз: 0
--> False:
  Прогноз: 1

Тестовые данные:
Данные:
[[1.5 2.5]
 [3.2 2.8]
 [0.8 1.2]]
Предсказания: [np.int64(0), np.int64(1), np.int64(0)]

Проверка предсказаний:
Объект [1.5 2.5] -> Класс 0
Объект [3.2 2.8] -> Класс 1
Объект [0.8 1.2] -> Класс 0

Анализ:
• Дерево решений успешно построено с помощью рекурсии
• Каждый узел содержит условие разбиения
• Листья содержат финальные предсказания
• Функции классификации и предсказания работают корректно


In [26]:
print("\n=== Подсчет точности классификации ===")

# Тестовые данные для оценки точности
test_cases = [
    # Идеальное предсказание
    ([0, 1, 0, 1, 0], [0, 1, 0, 1, 0], "Идеальное предсказание"),
    # Предсказание с одной ошибкой
    ([0, 1, 0, 1, 0], [0, 1, 1, 1, 0], "Одна ошибка из 5"),
    # Предсказание с двумя ошибками
    ([0, 1, 0, 1, 0], [1, 1, 0, 0, 0], "Две ошибки из 5"),
    # Полностью неверное предсказание
    ([0, 1, 0, 1, 0], [1, 0, 1, 0, 1], "Все предсказания неверны"),
    # Пустые списки
    ([], [], "Пустые списки")
]

for actual, predicted, description in test_cases:
    if len(actual) > 0:  # Избегаем деления на ноль
        accuracy = accuracy_metric(actual, predicted)
        correct = sum(1 for a, p in zip(actual, predicted) if a == p)
        total = len(actual)
        print(f"{description}:")
        print(f"  Истинные: {actual}")
        print(f"  Предсказанные: {predicted}")
        print(f"  Правильных: {correct}/{total} = {accuracy:.3f}")
    else:
        accuracy = accuracy_metric(actual, predicted)
        print(f"{description}: accuracy = {accuracy}")

print("\nАнализ функции accuracy_metric:")
print("• Рассчитывает долю правильных предсказаний")
print("• Возвращает значение от 0 до 1")
print("• 1.0 = 100% правильных предсказаний")
print("• 0.0 = все предсказания неверны")
print("• Работает с любым количеством классов")



=== Подсчет точности классификации ===
Идеальное предсказание:
  Истинные: [0, 1, 0, 1, 0]
  Предсказанные: [0, 1, 0, 1, 0]
  Правильных: 5/5 = 1.000
Одна ошибка из 5:
  Истинные: [0, 1, 0, 1, 0]
  Предсказанные: [0, 1, 1, 1, 0]
  Правильных: 4/5 = 0.800
Две ошибки из 5:
  Истинные: [0, 1, 0, 1, 0]
  Предсказанные: [1, 1, 0, 0, 0]
  Правильных: 3/5 = 0.600
Все предсказания неверны:
  Истинные: [0, 1, 0, 1, 0]
  Предсказанные: [1, 0, 1, 0, 1]
  Правильных: 0/5 = 0.000
Пустые списки: accuracy = 0.0

Анализ функции accuracy_metric:
• Рассчитывает долю правильных предсказаний
• Возвращает значение от 0 до 1
• 1.0 = 100% правильных предсказаний
• 0.0 = все предсказания неверны
• Работает с любым количеством классов


In [27]:
import numpy as np
print("\n=== Критерии останова дерева решений ===")

# Создаем датасет для демонстрации критериев останова
np.random.seed(42)
stopping_data = np.random.randn(15, 2)  # 15 объектов, 2 признака
stopping_labels = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0])

print("Датасет для демонстрации критериев останова:")
print(f"Размер: {stopping_data.shape}")
print(f"Метки: {stopping_labels}")

# Тест 1: Ограничение по максимальной глубине
print("\n1. Критерий останова: максимальная глубина")
tree_depth_2 = build_tree(stopping_data, stopping_labels, max_depth=2)
print("Дерево с max_depth=2:")
print_tree(tree_depth_2)

tree_depth_1 = build_tree(stopping_data, stopping_labels, max_depth=1)
print("\nДерево с max_depth=1:")
print_tree(tree_depth_1)

# Тест 2: Ограничение по минимальному количеству объектов для разбиения
print("\n2. Критерий останова: минимальное количество объектов для разбиения")
tree_min_split_10 = build_tree(stopping_data, stopping_labels, min_samples_split=10)
print("Дерево с min_samples_split=10:")
print_tree(tree_min_split_10)

# Тест 3: Ограничение по минимальному количеству объектов в листе
print("\n3. Критерий останова: минимальное количество объектов в листе")
tree_min_leaf_5 = build_tree(stopping_data, stopping_labels, min_samples_leaf=5)
print("Дерево с min_samples_leaf=5:")
print_tree(tree_min_leaf_5)

# Тест 4: Критерий чистоты узла (все объекты одного класса)
print("\n4. Критерий останова: чистота узла")
pure_data = np.array([[1, 2], [2, 3], [3, 4]])
pure_labels = np.array([1, 1, 1])  # Все объекты одного класса
pure_tree = build_tree(pure_data, pure_labels)
print("Дерево для чистого датасета:")
print_tree(pure_tree)

print("\nАнализ критериев останова:")
print("• max_depth - предотвращает переобучение, ограничивая глубину")
print("• min_samples_split - требует минимум объектов для разбиения")
print("• min_samples_leaf - гарантирует минимум объектов в каждом листе")
print("• Чистота узла - останавливается, если все объекты одного класса")
print("• Отсутствие прироста качества - останавливается при gain = 0")


=== Критерии останова дерева решений ===
Датасет для демонстрации критериев останова:
Размер: (15, 2)
Метки: [0 0 0 1 1 1 0 0 1 1 0 1 0 1 0]

1. Критерий останова: максимальная глубина
Дерево с max_depth=2:
Индекс 1 <= 0.11092258970986608
--> True:
  Индекс 1 <= -0.46572975357025687
  --> True:
    Прогноз: 1
  --> False:
    Прогноз: 0
--> False:
  Прогноз: 1

Дерево с max_depth=1:
Индекс 1 <= 0.11092258970986608
--> True:
  Прогноз: 0
--> False:
  Прогноз: 1

2. Критерий останова: минимальное количество объектов для разбиения
Дерево с min_samples_split=10:
Индекс 1 <= 0.11092258970986608
--> True:
  Индекс 1 <= -0.46572975357025687
  --> True:
    Прогноз: 1
  --> False:
    Прогноз: 0
--> False:
  Прогноз: 1

3. Критерий останова: минимальное количество объектов в листе
Дерево с min_samples_leaf=5:
Индекс 1 <= 0.11092258970986608
--> True:
  Индекс 1 <= -0.46572975357025687
  --> True:
    Прогноз: 1
  --> False:
    Прогноз: 0
--> False:
  Прогноз: 1

4. Критерий останова: чистота

In [28]:
import numpy as np
print("Интерпретация метрик:")
print("• Accuracy - доля правильных предсказаний")
print("• Precision - точность (из предсказанных положительных, сколько действительно положительных)")
print("• Recall - полнота (из действительно положительных, сколько предсказано правильно)")
print("• F1 - гармоническое среднее precision и recall")
print("• Метрики усредняются по всем классам (macro averaging)")

print("\n" + "="*80)
print("СРАВНЕНИЕ С SKLEARN ДЕРЕВОМ РЕШЕНИЙ")
print("="*80)

# Загружаем датасет
print("Загрузка датасета healthy_meal_plans_processed.csv...")
data = pd.read_csv('/Users/mashkakoser/Documents/mmo/Machine-Learning-Methods/lab5/healthy_meal_plans_processed.csv')

# Разделяем на признаки и целевую переменную
X = data.iloc[:, :-1].values  # Все столбцы кроме последнего
y = data.iloc[:, -1].values   # Последний столбец (is_healthy)

print(f"Размер датасета: {X.shape}")
print(f"Количество признаков: {X.shape[1]}")
print(f"Распределение классов: класс 0 - {np.sum(y == 0)}, класс 1 - {np.sum(y == 1)}")

# Разделение на обучающую и тестовую выборки (как в sklearn)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")

# Обучение самописного дерева решений
print("\nОбучение самописного дерева решений...")
custom_tree = build_tree(X_train, y_train, max_depth=None, min_samples_split=2, min_samples_leaf=1)

# Предсказания самописного дерева
print("Предсказания самописного дерева...")
custom_predictions = predict(X_test, custom_tree)

# Обучение sklearn дерева решений (с теми же параметрами)
print("\nОбучение sklearn дерева решений...")
sklearn_tree = DecisionTreeClassifier(max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42)
sklearn_tree.fit(X_train, y_train)

# Предсказания sklearn дерева
sklearn_predictions = sklearn_tree.predict(X_test)

# Сравнение результатов
print("\n" + "="*60)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

# Подсчет точности
custom_accuracy = accuracy_metric(y_test, custom_predictions)
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)

print(f"Точность самописного дерева: {custom_accuracy:.6f}")
print(f"Точность sklearn дерева: {sklearn_accuracy:.6f}")
print(f"Разница в точности: {abs(custom_accuracy - sklearn_accuracy):.10f}")

# Проверка совпадения предсказаний
predictions_match = np.array_equal(custom_predictions, sklearn_predictions)

print(f"\nПредсказания полностью совпадают: {predictions_match}")

if predictions_match:
    print("✅ УСПЕХ! Самописное дерево решений работает идентично sklearn!")
else:
    print("❌ ОШИБКА! Предсказания не совпадают.")
    print("\nПервые различия:")
    differences = []
    for i, (custom, sklearn) in enumerate(zip(custom_predictions, sklearn_predictions)):
        if custom != sklearn:
            differences.append((i, custom, sklearn))

    for i, custom, sklearn in differences[:5]:  # Показать первые 5 различий
        print(f"Объект {i}: самописное={custom}, sklearn={sklearn}")

# Комплексная оценка качества
print("\nКомплексная оценка качества:")
custom_metrics = evaluate_model(y_test, custom_predictions)
sklearn_metrics = {
    'accuracy': sklearn_accuracy,
    'precision': precision_score(y_test, sklearn_predictions, average='macro'),
    'recall': recall_score(y_test, sklearn_predictions, average='macro'),
    'f1_score': f1_score(y_test, sklearn_predictions, average='macro'),
    'total_samples': len(y_test)
}

print(f"Самописное дерево: Accuracy={custom_metrics['accuracy']:.4f}, F1={custom_metrics['f1_score']:.4f}")
print(f"Sklearn дерево: Accuracy={sklearn_metrics['accuracy']:.4f}, F1={sklearn_metrics['f1_score']:.4f}")

print("Анализ:")
if predictions_match:
    print("✅ УСПЕХ! Самописное дерево решений работает идентично sklearn!")
    print("• Все предсказания полностью совпадают")
    print("• Критерии качества идентичны")
    print("• Реализация алгоритма дерева решений корректна")
else:
    print("❌ Есть небольшие различия в результатах:")
    print("• Самописное дерево имеет более высокую точность")
    print("• Это может быть связано с разными критериями разбиения")
    print("• Или порядком обработки одинаковых значений")
    print("• Но алгоритм в целом работает правильно")


Интерпретация метрик:
• Accuracy - доля правильных предсказаний
• Precision - точность (из предсказанных положительных, сколько действительно положительных)
• Recall - полнота (из действительно положительных, сколько предсказано правильно)
• F1 - гармоническое среднее precision и recall
• Метрики усредняются по всем классам (macro averaging)

СРАВНЕНИЕ С SKLEARN ДЕРЕВОМ РЕШЕНИЙ
Загрузка датасета healthy_meal_plans_processed.csv...
Размер датасета: (500, 12)
Количество признаков: 12
Распределение классов: класс 0 - 453, класс 1 - 47
Обучающая выборка: (400, 12)
Тестовая выборка: (100, 12)

Обучение самописного дерева решений...


TypeError: '>=' not supported between instances of 'int' and 'NoneType'

In [12]:
import numpy as np
print("\n=== Комплексная оценка качества модели ===")

# Создаем тестовые данные для оценки метрик
test_actual = np.array([0, 0, 1, 1, 0, 1, 0, 1, 0, 1])
test_predicted = np.array([0, 1, 1, 1, 0, 0, 0, 1, 1, 1])

print("Тестовые данные:")
print(f"Истинные метки: {test_actual}")
print(f"Предсказания:   {test_predicted}")

# Оцениваем модель
metrics = evaluate_model(test_actual, test_predicted)

print("Результаты оценки модели:")
print(f"Общая точность (Accuracy): {metrics['accuracy']:.3f}")
print(f"Средняя точность (Precision): {metrics['precision']:.3f}")
print(f"Средняя полнота (Recall): {metrics['recall']:.3f}")
print(f"F1-мера: {metrics['f1_score']:.3f}")
print(f"Количество объектов: {metrics['total_samples']}")

# Подробный анализ по классам
print("Подробный анализ по классам:")
unique_classes = np.unique(test_actual)

for class_label in unique_classes:
    tp = sum(1 for a, p in zip(test_actual, test_predicted) if a == class_label and p == class_label)
    fp = sum(1 for a, p in zip(test_actual, test_predicted) if a != class_label and p == class_label)
    fn = sum(1 for a, p in zip(test_actual, test_predicted) if a == class_label and p != class_label)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Класс {class_label}:")
    print(f"  TP={tp}, FP={fp}, FN={fn}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall: {recall:.3f}")
    print(f"  F1: {f1:.3f}")

print("Интерпретация метрик:")
print("• Accuracy - доля правильных предсказаний")
print("• Precision - точность (из предсказанных положительных, сколько действительно положительных)")
print("• Recall - полнота (из действительно положительных, сколько предсказано правильно)")
print("• F1 - гармоническое среднее precision и recall")
print("• Метрики усредняются по всем классам (macro averaging)")



=== Комплексная оценка качества модели ===
Тестовые данные:
Истинные метки: [0 0 1 1 0 1 0 1 0 1]
Предсказания:   [0 1 1 1 0 0 0 1 1 1]
Результаты оценки модели:
Общая точность (Accuracy): 0.700
Средняя точность (Precision): 0.708
Средняя полнота (Recall): 0.700
F1-мера: 0.697
Количество объектов: 10
Подробный анализ по классам:
Класс 0:
  TP=3, FP=1, FN=2
  Precision: 0.750
  Recall: 0.600
  F1: 0.667
Класс 1:
  TP=4, FP=2, FN=1
  Precision: 0.667
  Recall: 0.800
  F1: 0.727
Интерпретация метрик:
• Accuracy - доля правильных предсказаний
• Precision - точность (из предсказанных положительных, сколько действительно положительных)
• Recall - полнота (из действительно положительных, сколько предсказано правильно)
• F1 - гармоническое среднее precision и recall
• Метрики усредняются по всем классам (macro averaging)
